# Run a test client

## Apply the feature store definitions

We'll mount the `feature_store.yaml` ConfigMap created by the Operator within a Kubernetes `Deployment` to run `feast apply`.

Then we create the client `Deployment` to apply the definitions, according to the [client.yaml](./client.yaml) manifest

We are going to emulate the `feast init -t postgres sample` command using Python code in an initContainer at client `Deployment` runtime. This is needed to mock the request of additional
parameters to configure the DB connection and also request the upload of example data to Postgres tables.

In [1]:
!kubectl apply -f client.yaml

Monitoring the log of the `Deployment` and checking DB for populated data.

In [1]:
!kubectl wait --for=condition=available deployment/client --timeout=2m
!kubectl logs deploy/client -c feast-apply
!kubectl exec deploy/postgres -- psql -h localhost -U feast feast -c 'select count(*) from sample_driver_hourly_stats'

deployment.apps/client condition met
Starting feast initialization job...

Creating a new Feast repository in /feast-init/sample.


Executing feast apply...
/opt/conda/lib/python3.11/site-packages/feast/feature_store.py:575: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(
No project found in the repository. Using project name sample defined in feature_store.yaml
Applying changes for project sample
Deploying infrastructure for driver_hourly_stats_fresh
Deploying infrastructure for driver_hourly_stats
Materializing 2 feature views to 2024-12-12 16:05:01+00:00 into the postgres online store.

driver_hourly_stats from 2024-12-12 15:54:27+00:00 to 2024-12-12 16:05:01+00:00:
0it [00:00, ?it/s]
driver_hourly_stats_fresh from 2024-12-12 15:54:27+00:00 to 2024-12-12 16:05:01+00:00:
0it [00:00, ?it/s]
 count 
-------
    15
(1 row)



Verify the client `feature_store.yaml`.

In [4]:
!kubectl exec deploy/client -c client -- cat feature_store.yaml

project: sample
provider: local
offline_store:
    host: feast-example-offline.feast.svc.cluster.local
    type: remote
    port: 80
online_store:
    path: http://feast-example-online.feast.svc.cluster.local:80
    type: remote
registry:
    path: feast-example-registry.feast.svc.cluster.local:80
    registry_type: remote
auth:
    type: no_auth
entity_key_serialization_version: 3


## Execute feast commands inside the client Pod

Finally, we run the full test suite from the client folder.

In [2]:
!kubectl exec deploy/client -c client -- feast feature-views list
!kubectl exec deploy/client -c client -- feast feature-views describe driver_hourly_stats
!kubectl exec deploy/client -c client -- feast entities list
!kubectl exec deploy/client -c client -- feast data-sources list

NAME                         ENTITIES    TYPE
driver_hourly_stats_fresh    {'driver'}  FeatureView
driver_hourly_stats          {'driver'}  FeatureView
transformed_conv_rate        {'driver'}  OnDemandFeatureView
transformed_conv_rate_fresh  {'driver'}  OnDemandFeatureView
spec:
  name: driver_hourly_stats
  entities:
  - driver
  features:
  - name: conv_rate
    valueType: FLOAT
  - name: acc_rate
    valueType: FLOAT
  - name: avg_daily_trips
    valueType: INT64
  tags:
    team: driver_performance
  ttl: 86400s
  batchSource:
    type: CUSTOM_SOURCE
    timestampField: event_timestamp
    createdTimestampColumn: created
    customOptions:
      configuration: eyJuYW1lIjogImRyaXZlcl9ob3VybHlfc3RhdHNfc291cmNlIiwgInF1ZXJ5IjogIlNFTEVDVCAqIEZST00gZmVhc3RfZHJpdmVyX2hvdXJseV9zdGF0cyIsICJ0YWJsZSI6ICIifQ==
    dataSourceClassType: feast.infra.offline_stores.contrib.postgres_offline_store.postgres_source.PostgreSQLSource
    name: driver_hourly_stats_source
  online: true
  entityColumns:
 

## Run test code inside the client Pod

Finally, we run the full test suite from the client folder.

In [3]:
!kubectl exec deploy/client -c client -- python test_workflow.py


--- Historical features for training ---
   driver_id     event_timestamp  ...  conv_rate_plus_val1  conv_rate_plus_val2
0       1001 2021-04-12 10:59:42  ...             1.316282            10.316282
1       1002 2021-04-12 08:12:10  ...             2.242124            20.242124
2       1003 2021-04-12 16:40:26  ...             3.954626            30.954626

[3 rows x 10 columns]

--- Historical features for batch scoring ---
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.096575
1       1002  ...           20.317592
2       1003  ...           30.021300

[3 rows x 10 columns]

--- Online features ---
acc_rate  :  [0.8634641170501709, 0.3706987798213959]
conv_rate_plus_val1  :  [1000.3753104507923, 1001.0143152652308]
conv_rate_plus_val2  :  [2000.3753104507923, 2002.0143152652308]
driver_id  :  [1001, 1002]

--- Online features retrieved (instead) through a feature service---
conv_rate  :  [0.3753104507923126, 0.014315265230834484]
conv_rate_plus_val1  :  [1000.

## Run feast python code inside the client Pod

Finally, we run the full test suite from the client folder.

In [5]:
!kubectl exec deploy/client -c client -- cat << EOF > blah.py
from pprint import pprint
from feast import FeatureStore

store = FeatureStore(repo_path=".")

feature_vector = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
    entity_rows=[
        # {join_key: entity_value}
        {"driver_id": 1004},
        {"driver_id": 1005},
    ],
).to_dict()

pprint(feature_vector)
EOF

!kubectl exec deploy/client -c client -- python blah.py

FileNotFoundError: [Errno 2] No such file or directory: 'feature_store.yaml'